# CLIP-ViT-B/16 + Transformer baseline trên Flickr30k

Bản chuyển từ `clipvit-transformer-coco2014.ipynb` sang **Flickr30k (Karpathy split)**, kèm cell **đánh giá trên tập test sau khi train xong**.

- Kiến trúc, hàm mất mát và vòng huấn luyện giữ nguyên: CLIP ViT-B/16 đóng băng + adapter, bộ giải mã Transformer 2 lớp, AdamW, AMP, EarlyStopping theo val loss.
- Dữ liệu: `dataset_flickr30k.json` (train 29.000 / val 1.014 / test 1.000 ảnh, 5 câu mỗi ảnh), không có restval. Cần gắn 2 dataset (Karpathy split Flickr30k và ảnh Flickr30k); đường dẫn tự dò dưới `/kaggle/input`.
- `EPOCHS` = 30 (mỗi epoch 1.512 bước, ít hơn COCO khoảng 3,9 lần).
- Tên checkpoint/từ điển có tiền tố `flickr30k_karpathy_baseline_`.


In [ ]:
import os
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
from transformers import CLIPVisionModel, get_linear_schedule_with_warmup
import json # Thêm json để xử lý COCO annotation và lưu vocab

# ==========================================
# CẤU HÌNH VÀ SIÊU THAM SỐ
# ==========================================
SEED = 42

def seed_everything(seed=SEED):
    """Cố định RNG để các lần train có thể tái lập."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # fixed IMAGE_SIZE/BATCH_SIZE -> an toan bat autotune cuDNN, tang toc

def seed_worker(worker_id):
    """Seed Python/NumPy trong từng DataLoader worker."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn Flickr30k trên Kaggle. None = tự dò dưới /kaggle/input, nên chỉ cần gắn (Add Input) 2 dataset:
#   1) Karpathy split của Flickr30k (file dataset_flickr30k.json: train 29.000 / val 1.014 / test 1.000 ảnh, 5 câu mỗi ảnh),
#   2) ảnh Flickr30k (thư mục chứa các file .jpg, vd 1000092795.jpg).
# Có thể điền tay nếu dò sai.
KARPATHY_JSON_FILE = None
FLICKR_IMG_DIR = None
# Dự phòng khi không có file Karpathy: captions.txt (cột image,caption) hoặc results.csv (image_name| comment_number| comment).
# Khi đó ảnh được chia ngẫu nhiên (seed cố định) 29.000 / 1.014 / 1.000, KHÔNG trùng split Karpathy trong các bài báo.
FLICKR_CAPTIONS_FILE = None
# Flickr30k Karpathy không có restval: tập huấn luyện chỉ gồm "train".
TRAIN_SPLITS = ("train",)

# Đường dẫn lưu trữ cấu hình sinh ra trong quá trình huấn luyện
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_baseline_vocab.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_baseline_last_checkpoint.pth")
BEST_MODEL_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_baseline_best_model.pth")
# Train từ đầu. Nếu kernel bị ngắt giữa chừng, đổi RESUME_TRAINING=True để tiếp tục từ last_checkpoint của chính bản này.
RESUME_TRAINING = False
RESUME_CHECKPOINT_FILE = CHECKPOINT_FILE

IMAGE_SIZE = (224, 224) 
SEQ_LENGTH = 25
MIN_WORD_FREQ = 5    
EMBED_DIM = 512
FF_DIM = 512
BATCH_SIZE = 96  # 32 -> 96: GPU nhanh hơn ~1,5 lần/mẫu, đỉnh bộ nhớ vài GB với AMP, Kaggle 16 GB dư sức; số bước/epoch giảm 3 lần.
EPOCHS = 30  # 1.512 bước/epoch (29.000 ảnh x 5 câu / 96); EarlyStopping (patience 3, val loss) thường dừng sớm hơn
NUM_HEADS = 8
NUM_LAYERS = 2
USE_AMP = True  # mixed precision (fp16 autocast + GradScaler) cho vong train - chi tang toc, KHONG dung cho vong val de giu metric val nguyen fp32

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
def custom_standardization(text):
    text = text.lower()
    strip_chars = r"""!"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"""
    text = re.sub(f"[{re.escape(strip_chars)}]", "", text)
    return text.strip()

# Train: Có Augmentation
train_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=36),
    T.ColorJitter(contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                std=[0.26862954, 0.26130258, 0.27577711])
])

# Validation: KHÔNG Augmentation
val_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], 
                std=[0.26862954, 0.26130258, 0.27577711])
])

class FlickrDataset(Dataset):
    def __init__(self, image_paths, captions, vocab, transforms=None):
        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        caption = self.captions[idx]
        
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback nếu ảnh bị lỗi (Rất hiếm nhưng có thể xảy ra ở bộ dữ liệu lớn)
            image = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))
            
        if self.transforms:
            image = self.transforms(image)
            
        tokens = [self.vocab.get(word, UNK_IDX) for word in caption.split()]
        tokens = [START_IDX] + tokens[:SEQ_LENGTH-2] + [END_IDX]
        
        padded_tokens = tokens + [PAD_IDX] * (SEQ_LENGTH - len(tokens))
        target_tensor = torch.tensor(padded_tokens, dtype=torch.long)
        
        return image, target_tensor

# ==========================================
# DỮ LIỆU FLICKR30K (Karpathy split)
# ==========================================
_FLICKR_META = None   # list[(filename, split, [câu gốc])]
SPLIT_SOURCE = None


def _find_first(root, predicate):
    for dirpath, _, files in os.walk(root):
        for name in files:
            if predicate(name):
                return os.path.join(dirpath, name)
    return None


def _load_flickr_meta():
    """Đọc danh sách (filename, split, câu gốc) một lần: ưu tiên dataset_flickr30k.json, nếu không có thì dùng file caption + chia ngẫu nhiên."""
    global _FLICKR_META, SPLIT_SOURCE, KARPATHY_JSON_FILE, FLICKR_IMG_DIR, FLICKR_CAPTIONS_FILE
    if _FLICKR_META is not None:
        return _FLICKR_META

    if KARPATHY_JSON_FILE is None and FLICKR_CAPTIONS_FILE is None:
        KARPATHY_JSON_FILE = _find_first("/kaggle/input", lambda n: n == "dataset_flickr30k.json")
        if KARPATHY_JSON_FILE is None:
            FLICKR_CAPTIONS_FILE = (_find_first("/kaggle/input", lambda n: n == "captions.txt")
                                    or _find_first("/kaggle/input", lambda n: n == "results.csv"))
    meta = []
    if KARPATHY_JSON_FILE:
        with open(KARPATHY_JSON_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        for img in data.get("images", []):
            sents = [s["raw"] if "raw" in s else " ".join(s.get("tokens", [])) for s in img.get("sentences", [])]
            if img.get("filename") and sents:
                meta.append((img["filename"], img.get("split"), sents))
        SPLIT_SOURCE = f"Karpathy split ({KARPATHY_JSON_FILE})"
    elif FLICKR_CAPTIONS_FILE:
        by_image = {}
        with open(FLICKR_CAPTIONS_FILE, "r", encoding="utf-8") as f:
            header = f.readline()
            sep = "|" if "|" in header else ","
            for line in f:
                parts = line.rstrip("\n").split(sep, 2 if sep == "|" else 1)
                if len(parts) < 2:
                    continue
                name, cap = parts[0].strip(), parts[-1].strip()
                by_image.setdefault(name, []).append(cap)
        names = sorted(by_image)
        random.Random(SEED).shuffle(names)
        for i, name in enumerate(names):
            split = "train" if i < 29000 else "val" if i < 29000 + 1014 else "test" if i < 29000 + 1014 + 1000 else None
            if split:
                meta.append((name, split, by_image[name]))
        SPLIT_SOURCE = f"CHIA NGẪU NHIÊN từ {FLICKR_CAPTIONS_FILE} (không phải split Karpathy)"
    else:
        raise RuntimeError("Không tìm thấy dataset_flickr30k.json hay captions.txt/results.csv dưới /kaggle/input. "
                           "Hãy gắn dataset Flickr30k hoặc điền KARPATHY_JSON_FILE / FLICKR_CAPTIONS_FILE.")

    if FLICKR_IMG_DIR is None:
        probe = meta[0][0]
        hit = _find_first("/kaggle/input", lambda n: n == probe)
        if hit is None:
            raise RuntimeError(f"Không thấy ảnh {probe} dưới /kaggle/input. Hãy gắn dataset ảnh Flickr30k hoặc điền FLICKR_IMG_DIR.")
        FLICKR_IMG_DIR = os.path.dirname(hit)
    _FLICKR_META = meta
    return meta


def load_flickr_data(split="train"):
    """Trả về (image_paths, captions) theo cặp ảnh-caption; split là một tên hoặc tuple tên."""
    splits = {split} if isinstance(split, str) else set(split)
    image_paths, captions = [], []
    missing = 0
    for filename, sp, sents in _load_flickr_meta():
        if sp not in splits:
            continue
        path = os.path.join(FLICKR_IMG_DIR, filename)
        if not os.path.exists(path):
            missing += 1
            continue
        for raw in sents:
            caption = custom_standardization(raw)
            if caption:
                image_paths.append(path)
                captions.append(caption)
    print(f"-> Flickr30k {sorted(splits)}: {len(set(image_paths))} ảnh, {len(captions)} cặp ảnh-caption.")
    if missing:
        print(f"-> Bỏ qua {missing} ảnh vì không tìm thấy file ảnh.")
    return image_paths, captions


def build_or_load_vocab(captions, min_freq=5, vocab_file=VOCAB_FILE):
    """Hàm tạo Vocabulary mới hoặc tải từ file nếu đã tồn tại"""
    if os.path.exists(vocab_file):
        print(f"-> Tìm thấy file Vocab. Đang tải Vocabulary từ {vocab_file}...")
        with open(vocab_file, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"-> Kích thước Vocabulary đã tải: {len(vocab)}")
        return vocab

    print(f"-> Không tìm thấy file {vocab_file}. Đang xây dựng Vocabulary mới...")
    all_words = []
    for cap in captions:
        all_words.extend(cap.split())
        
    word_counts = Counter(all_words)
    valid_words = [word for word, count in word_counts.items() if count >= min_freq]
    
    vocab = {"<pad>": PAD_IDX, "<start>": START_IDX, "<end>": END_IDX, "<unk>": UNK_IDX}
    for idx, word in enumerate(valid_words):
        vocab[word] = idx + 4 
        
    # Lưu lại Vocab để tái sử dụng
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)
        
    print(f"-> Kích thước Vocabulary (từ xuất hiện >= {min_freq} lần): {len(vocab)}")
    print(f"-> Đã lưu Vocabulary tại: {vocab_file}")
    return vocab

# ==========================================
# KIẾN TRÚC MÔ HÌNH
# ==========================================
class CLIPEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPEncoder, self).__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch16",
            use_safetensors=True,
        )
        
        for param in self.clip_vision.parameters():
            param.requires_grad = False
            
        self.fc = nn.Linear(768, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        outputs = self.clip_vision(pixel_values=images)
        features = outputs.last_hidden_state 
        embeddings = self.fc(features) 
        return self.layer_norm(embeddings)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class ImageCaptioningTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, ff_dim, num_heads, num_layers):
        super(ImageCaptioningTransformer, self).__init__()
        self.encoder = CLIPEncoder(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, images, captions):
        memory = self.encoder(images) 
        tgt_embed = self.embedding(captions) 
        tgt_embed = self.pos_encoder(tgt_embed)
        tgt_mask = self.generate_square_subsequent_mask(captions.size(1)).to(images.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        out = self.transformer_decoder(tgt=tgt_embed, memory=memory, 
                                       tgt_mask=tgt_mask, 
                                       tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(out)

def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _prepare_save_path(filepath):
    """Đảm bảo checkpoint được lưu vào thư mục ghi được, không phải /kaggle/input."""
    if os.path.isdir(filepath):
        raise RuntimeError(
            f"'{filepath}' là thư mục, không phải file checkpoint. "
            "Hãy truyền CHECKPOINT_FILE hoặc BEST_MODEL_FILE, ví dụ /kaggle/working/last_checkpoint.pth."
        )
    save_dir = os.path.dirname(filepath) or "."
    os.makedirs(save_dir, exist_ok=True)
    if not os.access(save_dir, os.W_OK):
        raise RuntimeError(
            f"Không thể ghi checkpoint vào '{save_dir}'. "
            "Trên Kaggle, hãy lưu vào /kaggle/working và chỉ load checkpoint từ /kaggle/input."
        )
    return filepath

def _resolve_checkpoint_path(filepath):
    """Cho phép RESUME_CHECKPOINT_FILE là file .pth hoặc folder chứa checkpoint."""
    if os.path.isfile(filepath):
        return filepath

    if os.path.isdir(filepath):
        preferred_names = ["last_checkpoint.pth", "best_model.pth"]
        for name in preferred_names:
            candidate = os.path.join(filepath, name)
            if os.path.isfile(candidate):
                return candidate

        checkpoint_files = []
        for root, _, files in os.walk(filepath):
            for name in files:
                if name.endswith((".pth", ".pt")):
                    checkpoint_files.append(os.path.join(root, name))

        if checkpoint_files:
            checkpoint_files.sort(key=os.path.getmtime, reverse=True)
            return checkpoint_files[0]

        raise RuntimeError(f"'{filepath}' là thư mục nhưng không tìm thấy file .pth/.pt bên trong.")

    return filepath

class EarlyStoppingCallback:
    def __init__(self, patience=3, min_delta=0.0, save_path=BEST_MODEL_FILE):
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model, optimizer, epoch):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_best_model(model, optimizer, epoch)
        else:
            self.counter += 1
            print(f" EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_best_model(self, model, optimizer, epoch):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': _unwrap_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': self.best_loss
        }
        save_path = _prepare_save_path(self.save_path)
        torch.save(checkpoint, save_path)
        print(f" => Val Loss giảm. Đã lưu BEST MODEL tại: {save_path}")

# ==========================================
# QUẢN LÝ CHECKPOINT VÀ HUẤN LUYỆN
# ==========================================
def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, early_stopping, filepath=CHECKPOINT_FILE):
    """Lưu Checkpoint bao gồm cả trạng thái Early Stopping"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'val_loss': val_loss,
        'best_loss': early_stopping.best_loss,
        'es_counter': early_stopping.counter
    }
    filepath = _prepare_save_path(filepath)
    torch.save(checkpoint, filepath)
    print(f" => Đã lưu Checkpoint phục hồi tại: {filepath}")

def load_checkpoint(model, optimizer=None, scheduler=None, early_stopping=None, filepath=CHECKPOINT_FILE, resume_training=True):
    """Tải checkpoint để train tiếp; nếu thiếu optimizer/scheduler thì chỉ load trọng số model."""
    start_epoch = 0
    if not resume_training:
        print("[*] RESUME_TRAINING = False. Sẽ bắt đầu huấn luyện từ Epoch 1.")
        return start_epoch

    filepath = _resolve_checkpoint_path(filepath)

    if os.path.exists(filepath):
        print(f"[*] Tìm thấy Checkpoint '{filepath}'. Đang tải...")
        checkpoint = torch.load(filepath, map_location=device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        _unwrap_model(model).load_state_dict(state_dict)

        if optimizer is not None and 'optimizer_state_dict' in checkpoint:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

        start_epoch = checkpoint.get('epoch', -1) + 1
        
        # Khôi phục trạng thái Early Stopping
        if early_stopping is not None:
            early_stopping.best_loss = checkpoint.get('best_loss', float('inf'))
            early_stopping.counter = checkpoint.get('es_counter', 0)
        
        print(f"[*] Đã khôi phục thành công. Sẽ tiếp tục huấn luyện từ Epoch {start_epoch + 1}")
        if early_stopping is not None:
            print(f"[*] Trạng thái EarlyStopping: best_loss = {early_stopping.best_loss:.4f}, counter = {early_stopping.counter}")
    else:
        print(f"[*] Không tìm thấy Checkpoint '{filepath}'. Sẽ bắt đầu huấn luyện từ Epoch 1.")
    return start_epoch

def train_model(train_loader, val_loader, vocab_size):
    model = ImageCaptioningTransformer(
        vocab_size=vocab_size, embed_dim=EMBED_DIM, 
        ff_dim=FF_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS
    ).to(device)
    
    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"[*] Dùng {torch.cuda.device_count()} GPU với nn.DataParallel")
        model = nn.DataParallel(model)
        
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=0.01)
    
    num_training_steps = len(train_loader) * EPOCHS
    num_warmup_steps = int(0.05 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    
    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    print(f" Mixed precision (AMP): {'BAT' if amp_enabled else 'TAT'} (USE_AMP={USE_AMP}, cuda={torch.cuda.is_available()})")

    early_stopping = EarlyStoppingCallback(patience=3, save_path=BEST_MODEL_FILE)
    
    # -------------------------
    # RESUME TỪ CHECKPOINT
    # -------------------------
    start_epoch = load_checkpoint(
        model,
        optimizer,
        scheduler,
        early_stopping,
        filepath=RESUME_CHECKPOINT_FILE,
        resume_training=RESUME_TRAINING
    )
    
    log_interval = 200 # Bộ dữ liệu lớn, in thưa hơn để tránh trôi màn hình
    
    for epoch in range(start_epoch, EPOCHS):
        # Seed theo epoch giúp resume từ checkpoint vẫn tái lập đúng thứ tự batch/augmentation.
        epoch_seed = SEED + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        print(f"\n--- Bắt đầu Epoch {epoch+1:02d}/{EPOCHS} ---")
        
        # -------------------------
        # PHASE 1: TRAINING
        # -------------------------
        model.train()
        train_loss = 0
        train_correct = 0
        train_tokens = 0
        
        for batch_idx, (images, captions) in enumerate(train_loader):
            images, captions = images.to(device), captions.to(device)
            
            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]
            
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                outputs = model(images, decoder_input)
                loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            
            predictions = outputs.argmax(dim=-1) 
            mask = (targets != PAD_IDX) 
            correct = (predictions == targets) & mask
            
            train_correct += correct.sum().item()
            train_tokens += mask.sum().item()
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            
            if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
                batch_acc = (correct.sum().item() / mask.sum().item()) * 100 if mask.sum().item() > 0 else 0.0
                print(f"   [Epoch {epoch+1:02d}] Batch {batch_idx+1:05d}/{len(train_loader)} | Batch Loss: {loss.item():.4f} | Batch Acc: {batch_acc:.2f}%")
                
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = (train_correct / train_tokens) * 100
        
        # -------------------------
        # PHASE 2: VALIDATION
        # -------------------------
        model.eval()
        val_loss = 0
        val_correct = 0
        val_tokens = 0

        if val_loader.generator is not None:
            val_loader.generator.manual_seed(SEED + 100_000 + epoch)
        
        with torch.no_grad():
            for images, captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                
                decoder_input = captions[:, :-1]
                targets = captions[:, 1:]
                
                outputs = model(images, decoder_input)
                loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                val_loss += loss.item()
                
                predictions = outputs.argmax(dim=-1) 
                mask = (targets != PAD_IDX) 
                val_correct += ((predictions == targets) & mask).sum().item()
                val_tokens += mask.sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_tokens) * 100

        # -------------------------
        # BÁO CÁO VÀ LƯU CHECKPOINT
        # -------------------------
        print(f"\n=> TỔNG KẾT EPOCH {epoch+1:02d}:")
        print(f"   Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")
        print(f"   Val Loss  : {epoch_val_loss:.4f} | Val Acc  : {epoch_val_acc:.2f}%")
        
        # 1. Lưu Best Model nếu loss giảm
        early_stopping(epoch_val_loss, model, optimizer, epoch)
        
        # 2. Luôn lưu Last Checkpoint sau mỗi epoch để Resume khi sập Kaggle kernel
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_val_loss, early_stopping, CHECKPOINT_FILE)
        
        if early_stopping.early_stop:
            print(f"⚠️ Dừng huấn luyện sớm tại Epoch {epoch+1} do Val Loss không giảm liên tiếp.")
            break
            
    return model, optimizer

# ==========================================
# THỰC THI (MAIN)
# ==========================================
if __name__ == "__main__":
    print("0. Đang chuẩn bị Dataset và Vocabulary cho baseline Flickr30k...")
    
    # 0.1 Load danh sách ảnh và caption tập Train theo Karpathy split
    train_imgs, train_caps = load_flickr_data(TRAIN_SPLITS)
    print(f"   Nguồn split: {SPLIT_SOURCE} | thư mục ảnh: {FLICKR_IMG_DIR}")
    
    # 0.2 Load danh sách ảnh và caption tập Val theo Karpathy split
    val_imgs, val_caps = load_flickr_data("val")
    
    # 0.3 Xây dựng (hoặc Tải) Vocabulary từ tập Train và lưu ra file (đảm bảo file có thể load lại)
    vocab = build_or_load_vocab(train_caps, min_freq=MIN_WORD_FREQ, vocab_file=VOCAB_FILE)
    vocab_size = len(vocab)
    
    # 1. Khởi tạo Dataset & DataLoader
    train_dataset = FlickrDataset(train_imgs, train_caps, vocab, transforms=train_transforms)
    val_dataset = FlickrDataset(val_imgs, val_caps, vocab, transforms=val_transforms)

    print("1. Đang khởi tạo DataLoaders...")
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)
    val_generator = torch.Generator()
    val_generator.manual_seed(SEED + 1)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=val_generator
    )

    print(f"2. Bắt đầu huấn luyện baseline CLIP-ViT-B/16 + Transformer trên Flickr30k...")
    model, optimizer = train_model(train_loader, val_loader, vocab_size=vocab_size)

    print("3. Quá trình huấn luyện hoàn tất toàn bộ số epochs yêu cầu.")


## Đánh giá sau khi train (tập test Karpathy, 1.000 ảnh)

Cell kế tiếp nạp **best model** (`BEST_MODEL_FILE`), sinh câu cho tập test bằng greedy và beam search rồi tính:

- **Chất lượng câu**: BLEU-1..4, METEOR (cần Java), ROUGE-L, CIDEr; SPICE tùy chọn (`EVAL_USE_SPICE`, chậm, cần Java và tải Stanford CoreNLP).
- **Ảo giác đối tượng (CHAIRs / CHAIRi)**: dùng bảng đồng nghĩa CHAIR gốc (Rohrbach et al., 2018). Nhãn đúng lấy từ 5 caption tham chiếu của ảnh
  vì Flickr30k không có COCO instances, nên con số cao hơn và **không so trực tiếp được** với CHAIR trên COCO.

Kết quả lưu ở `EVAL_RESULT_FILE` (JSON). Nếu kernel đã khởi động lại, chạy lại các cell định nghĩa (cấu hình, dữ liệu, mô hình) rồi chạy cell này; cell tự nạp từ điển và checkpoint từ `/kaggle/working`.


In [ ]:
# ==========================================
# ĐÁNH GIÁ SAU KHI TRAIN: tập test Karpathy của Flickr30k
# ==========================================
import shutil
import time
import torch.nn.functional as F

EVAL_SPLIT = "test"
EVAL_CHECKPOINT = BEST_MODEL_FILE   # đổi thành CHECKPOINT_FILE để đánh giá epoch cuối
EVAL_BEAM_SIZE = 3
EVAL_LENGTH_ALPHA = 0.7             # chọn câu beam theo logprob / độ dài^alpha
EVAL_MAX_IMAGES = None              # None = toàn bộ; đặt số nhỏ (vd 50) để chạy thử nhanh
EVAL_BATCH_SIZE = 50
EVAL_USE_METEOR = True              # cần Java; thiếu Java thì tự bỏ qua
EVAL_USE_SPICE = False              # cần Java + tải Stanford CoreNLP, chậm (hàng chục phút với 1.000 ảnh)
EVAL_RESULT_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_baseline_eval_test.json")

CHAIR_SYNONYMS_TEXT = """person, girl, boy, man, woman, kid, child, chef, baker, people, adult, rider, children, baby, worker, passenger, sister, biker, policeman, cop, officer, lady, cowboy, bride, groom, male, female, guy, traveler, mother, father, gentleman, pitcher, player, skier, snowboarder, skater, skateboarder, person, woman, guy, foreigner, child, gentleman, caller, offender, coworker, trespasser, patient, politician, soldier, grandchild, serviceman, walker, drinker, doctor, bicyclist, thief, buyer, teenager, student, camper, driver, solider, hunter, shopper, villager
bicycle, bike, bicycle, bike, unicycle, minibike, trike
car, automobile, van, minivan, sedan, suv, hatchback, cab, jeep, coupe, taxicab, limo, taxi
motorcycle, scooter,  motor bike, motor cycle, motorbike, scooter, moped
airplane, jetliner, plane, air plane, monoplane, aircraft, jet, jetliner, airbus, biplane, seaplane
bus, minibus, trolley
train, locomotive, tramway, caboose
truck, pickup, lorry, hauler, firetruck
boat, ship, liner, sailboat, motorboat, dinghy, powerboat, speedboat, canoe, skiff, yacht, kayak, catamaran, pontoon, houseboat, vessel, rowboat, trawler, ferryboat, watercraft, tugboat, schooner, barge, ferry, sailboard, paddleboat, lifeboat, freighter, steamboat, riverboat, battleship, steamship
traffic light, street light, traffic signal, stop light, streetlight, stoplight
fire hydrant, hydrant
stop sign
parking meter
bench, pew
bird, ostrich, owl, seagull, goose, duck, parakeet, falcon, robin, pelican, waterfowl, heron, hummingbird, mallard, finch, pigeon, sparrow, seabird, osprey, blackbird, fowl, shorebird, woodpecker, egret, chickadee, quail, bluebird, kingfisher, buzzard, willet, gull, swan, bluejay, flamingo, cormorant, parrot, loon, gosling, waterbird, pheasant, rooster, sandpiper, crow, raven, turkey, oriole, cowbird, warbler, magpie, peacock, cockatiel, lorikeet, puffin, vulture, condor, macaw, peafowl, cockatoo, songbird
cat, kitten, feline, tabby
dog, puppy, beagle, pup, chihuahua, schnauzer, dachshund, rottweiler, canine, pitbull, collie, pug, terrier, poodle, labrador, doggie, doberman, mutt, doggy, spaniel, bulldog, sheepdog, weimaraner, corgi, cocker, greyhound, retriever, brindle, hound, whippet, husky
horse, colt, pony, racehorse, stallion, equine, mare, foal, palomino, mustang, clydesdale, bronc, bronco
sheep, lamb, ram, lamb, goat, ewe
cow, cattle, oxen, ox, calf, cattle, holstein, heifer, buffalo, bull, zebu, bison
elephant
bear, panda
zebra
giraffe
backpack, knapsack
umbrella
handbag, wallet, purse, briefcase
tie, bow, bow tie
suitcase, suit case, luggage
frisbee
skis, ski
snowboard
sports ball, ball
kite
baseball bat
baseball glove
skateboard
surfboard, longboard, skimboard, shortboard, wakeboard
tennis racket, racket
bottle
wine glass
cup
fork
knife, pocketknife, knive
spoon
bowl, container
banana
apple
sandwich, burger, sub, cheeseburger, hamburger
orange
broccoli
carrot
hot dog
pizza
donut, doughnut, bagel
cake,  cheesecake, cupcake, shortcake, coffeecake, pancake
chair, seat, stool
couch, sofa, recliner, futon, loveseat, settee, chesterfield
potted plant, houseplant
bed
dining table, table, desk
toilet, urinal, commode, toilet, lavatory, potty
tv, monitor, televison, television
laptop, computer, notebook, netbook, lenovo, macbook, laptop computer
mouse
remote
keyboard
cell phone, mobile phone, phone, cellphone, telephone, phon, smartphone, iPhone
microwave
oven, stovetop, stove, stove top oven
toaster
sink
refrigerator, fridge, fridge, freezer
book
clock
vase
scissors
teddy bear, teddybear
hair drier, hairdryer
toothbrush"""
_CHAIR_DOUBLE_WORDS = [
    "motor bike", "motor cycle", "air plane", "traffic light", "street light", "traffic signal",
    "stop light", "fire hydrant", "stop sign", "parking meter", "suit case", "sports ball",
    "baseball bat", "baseball glove", "tennis racket", "wine glass", "hot dog", "cell phone",
    "mobile phone", "teddy bear", "hair drier", "potted plant", "bow tie", "laptop computer",
    "stove top oven", "home plate", "train track",
]


def _load_chair_tables():
    objects, inverse = set(), {}
    for line in CHAIR_SYNONYMS_TEXT.splitlines():
        syns = [s.strip() for s in line.split(",") if s.strip()]
        if not syns:
            continue
        objects.update(syns)
        for s in syns:
            inverse[s] = syns[0]
    double = {w: w for w in _CHAIR_DOUBLE_WORDS}
    for w in ["bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "animal", "cub"]:
        double[f"baby {w}"] = w
        double[f"adult {w}"] = w
    for w in ["jet", "train"]:
        double[f"passenger {w}"] = w
    double.update({"bow tie": "tie", "toilet seat": "toilet", "wine glas": "wine glass"})
    return objects, inverse, double


_CHAIR_OBJECTS, _CHAIR_INVERSE, _CHAIR_DOUBLE = _load_chair_tables()
try:
    from nltk.stem import WordNetLemmatizer
    _LEMMA = WordNetLemmatizer()
    _LEMMA.lemmatize("dogs", pos="n")
except Exception:
    try:
        import nltk
        nltk.download("wordnet", quiet=True)
        from nltk.stem import WordNetLemmatizer
        _LEMMA = WordNetLemmatizer()
        _LEMMA.lemmatize("dogs", pos="n")
    except Exception:
        _LEMMA = None  # không có wordnet: dùng luật số nhiều đơn giản bên dưới


def _singular(word):
    if _LEMMA is not None:
        return _LEMMA.lemmatize(word, pos="n")
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("ses", "xes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss") and len(word) > 3:
        return word[:-1]
    return word


def chair_objects(caption):
    """Các lớp COCO (từ chuẩn) được caption nhắc tới, theo cách chair.py của CHAIR."""
    words = [_singular(w) for w in caption.lower().split()]
    merged, i = [], 0
    while i < len(words):
        two = " ".join(words[i:i + 2])
        if two in _CHAIR_DOUBLE:
            merged.append(_CHAIR_DOUBLE[two])
            i += 2
        else:
            merged.append(words[i])
            i += 1
    if "toilet" in merged and "seat" in merged:
        merged = [w for w in merged if w != "seat"]
    return [_CHAIR_INVERSE[w] for w in merged if w in _CHAIR_OBJECTS]


def compute_chair(preds, refs_list):
    """CHAIRs: % câu nhắc ít nhất 1 object không có trong 5 caption tham chiếu; CHAIRi: % lượt nhắc object bị ảo giác."""
    n_caps = n_hall_caps = n_mentions = n_hall = skipped = 0
    for pred, refs in zip(preds, refs_list):
        gt = set()
        for r in refs:
            gt.update(chair_objects(r))
        if not gt:
            skipped += 1
            continue
        mentioned = chair_objects(pred)
        bad = [o for o in mentioned if o not in gt]
        n_caps += 1
        n_mentions += len(mentioned)
        n_hall += len(bad)
        n_hall_caps += bool(bad)
    return {"CHAIRs": 100.0 * n_hall_caps / max(n_caps, 1), "CHAIRi": 100.0 * n_hall / max(n_mentions, 1),
            "num_captions": n_caps, "skipped_no_ground_truth": skipped}


def _ids_to_text(ids, idx2word):
    out = []
    for t in ids:
        if t == END_IDX:
            break
        if t in (START_IDX, PAD_IDX):
            continue
        out.append(idx2word.get(t, "<unk>"))
    return " ".join(out)


@torch.no_grad()
def decode_greedy(model_, images):
    memory = model_.encoder(images)
    B = images.size(0)
    seqs = torch.full((B, 1), START_IDX, dtype=torch.long, device=images.device)
    done = torch.zeros(B, dtype=torch.bool, device=images.device)
    for _ in range(SEQ_LENGTH - 1):
        mask = model_.generate_square_subsequent_mask(seqs.size(1)).to(images.device)
        out = model_.transformer_decoder(tgt=model_.pos_encoder(model_.embedding(seqs)), memory=memory, tgt_mask=mask)
        nxt = model_.fc_out(out[:, -1]).argmax(-1).masked_fill(done, PAD_IDX)
        seqs = torch.cat([seqs, nxt.unsqueeze(1)], dim=1)
        done = done | (nxt == END_IDX)
        if done.all():
            break
    return seqs.tolist()


@torch.no_grad()
def decode_beam(model_, images, beam=3, alpha=0.7):
    memory = model_.encoder(images)
    B, L, D = memory.shape
    mem = memory.unsqueeze(1).expand(-1, beam, -1, -1).reshape(B * beam, L, D)
    seqs = torch.full((B * beam, 1), START_IDX, dtype=torch.long, device=images.device)
    scores = torch.zeros(B, beam, device=images.device)
    scores[:, 1:] = -1e9   # bước đầu chỉ giữ 1 nhánh để các beam không trùng nhau
    done = torch.zeros(B * beam, dtype=torch.bool, device=images.device)
    base = (torch.arange(B, device=images.device) * beam).unsqueeze(1)
    for _ in range(SEQ_LENGTH - 1):
        mask = model_.generate_square_subsequent_mask(seqs.size(1)).to(images.device)
        out = model_.transformer_decoder(tgt=model_.pos_encoder(model_.embedding(seqs)), memory=mem, tgt_mask=mask)
        logp = F.log_softmax(model_.fc_out(out[:, -1]).float(), dim=-1)
        V = logp.size(-1)
        logp[done] = -1e9
        logp[done, PAD_IDX] = 0.0   # nhánh đã kết thúc chỉ được nối <pad>, không đổi điểm
        cand = (scores.reshape(-1, 1) + logp).reshape(B, beam * V)
        top_scores, top_idx = cand.topk(beam, dim=1)
        sel = (base + top_idx // V).reshape(-1)
        tok = (top_idx % V).reshape(-1)
        seqs = torch.cat([seqs[sel], tok.unsqueeze(1)], dim=1)
        done = done[sel] | (tok == END_IDX)
        scores = top_scores
        if done.all():
            break
    lengths = ((seqs != PAD_IDX) & (seqs != START_IDX)).sum(dim=1).clamp_min(1).float().reshape(B, beam)
    best = (scores / lengths.pow(alpha)).argmax(dim=1)
    return seqs.reshape(B, beam, -1)[torch.arange(B, device=images.device), best].tolist()


def compute_caption_metrics(preds, refs_list):
    """BLEU/METEOR/ROUGE-L/CIDEr(/SPICE) bằng pycocoevalcap trên chữ đã chuẩn hóa sẵn (không cần PTBTokenizer). Điểm nhân 100."""
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.rouge.rouge import Rouge
    gts = {i: refs for i, refs in enumerate(refs_list)}
    res = {i: [p if p else "<unk>"] for i, p in enumerate(preds)}
    metrics = {}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    for k in range(4):
        metrics[f"BLEU-{k + 1}"] = 100 * bleu[k]
    has_java = shutil.which("java") is not None
    if EVAL_USE_METEOR and has_java:
        try:
            from pycocoevalcap.meteor.meteor import Meteor
            metrics["METEOR"] = 100 * Meteor().compute_score(gts, res)[0]
        except Exception as exc:
            print(f"[!] Bỏ qua METEOR: {exc}")
    elif EVAL_USE_METEOR:
        print("[!] Không có Java trong PATH: bỏ qua METEOR.")
    metrics["ROUGE-L"] = 100 * Rouge().compute_score(gts, res)[0]
    metrics["CIDEr"] = 100 * Cider().compute_score(gts, res)[0]
    if EVAL_USE_SPICE and has_java:
        try:
            from pycocoevalcap.spice.spice import Spice
            metrics["SPICE"] = 100 * Spice().compute_score(gts, res)[0]
        except Exception as exc:
            print(f"[!] Bỏ qua SPICE: {exc}")
    return metrics


class _EvalDataset(Dataset):
    def __init__(self, paths, transforms):
        self.paths, self.transforms = paths, transforms

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))
        return self.transforms(img)


def load_eval_model(checkpoint_file, vocab_size):
    m = ImageCaptioningTransformer(vocab_size=vocab_size, embed_dim=EMBED_DIM, ff_dim=FF_DIM,
                                   num_heads=NUM_HEADS, num_layers=NUM_LAYERS).to(device)
    path = _resolve_checkpoint_path(checkpoint_file)
    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt.get("model_state_dict", ckpt))
    m.eval()
    print(f"[*] Đã nạp {path} (epoch {ckpt.get('epoch', -1) + 1 if isinstance(ckpt, dict) else '?'})")
    return m


def run_evaluation():
    eval_vocab = json.load(open(VOCAB_FILE, "r", encoding="utf-8"))
    idx2word = {i: w for w, i in eval_vocab.items()}
    paths, caps = load_flickr_data(EVAL_SPLIT)
    refs_by_path, order = {}, []
    for p, c in zip(paths, caps):
        if p not in refs_by_path:
            refs_by_path[p] = []
            order.append(p)
        refs_by_path[p].append(c)
    if EVAL_MAX_IMAGES:
        order = order[:EVAL_MAX_IMAGES]
    refs_list = [refs_by_path[p] for p in order]
    print(f"[*] Đánh giá {len(order)} ảnh của split '{EVAL_SPLIT}' bằng {os.path.basename(EVAL_CHECKPOINT)}")

    model_ = load_eval_model(EVAL_CHECKPOINT, len(eval_vocab))
    loader = DataLoader(_EvalDataset(order, val_transforms), batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)

    results, samples = {}, {}
    for name, fn in (("greedy", lambda x: decode_greedy(model_, x)),
                     (f"beam{EVAL_BEAM_SIZE}", lambda x: decode_beam(model_, x, EVAL_BEAM_SIZE, EVAL_LENGTH_ALPHA))):
        t0 = time.time()
        preds = []
        for images in loader:
            for ids in fn(images.to(device)):
                preds.append(_ids_to_text(ids, idx2word))
        decode_s = time.time() - t0
        metrics = compute_caption_metrics(preds, refs_list)
        metrics.update(compute_chair(preds, refs_list))
        metrics["mean_len"] = sum(len(p.split()) for p in preds) / max(len(preds), 1)
        metrics["unk_rate_%"] = 100 * sum(p.split().count("<unk>") for p in preds) / max(sum(len(p.split()) for p in preds), 1)
        metrics["decode_seconds"] = decode_s
        results[name] = {k: (round(v, 2) if isinstance(v, float) else v) for k, v in metrics.items()}
        samples[name] = [{"image": os.path.basename(p), "pred": c, "refs": r[:2]} for p, c, r in list(zip(order, preds, refs_list))[:10]]
        print(f"\n=== {name} ===")
        for k, v in results[name].items():
            print(f"   {k:>16}: {v}")

    out = {"checkpoint": EVAL_CHECKPOINT, "split": EVAL_SPLIT, "split_source": SPLIT_SOURCE, "num_images": len(order),
           "beam_size": EVAL_BEAM_SIZE, "results": results, "samples": samples}
    with open(EVAL_RESULT_FILE, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    print(f"\n[*] Đã lưu kết quả: {EVAL_RESULT_FILE}")
    return out


if __name__ == "__main__":
    eval_out = run_evaluation()
